In [5]:
import numpy as np
import pandas as pd
import datetime as dt
import random
import string
import hashlib
import json

#Loading original data
columns_name = ['id', 'date', 'latitude', 'longitude' ]

df = pd.read_csv('../Data_Anonymisation_et_Reidentification/test_dataset.csv', names = columns_name, sep = '\t')
df['date'] = df['date'].str.slice(stop = 16)
df['date'] = pd.to_datetime(df['date'], format = "%Y-%m-%d %H:%M")
df['week'] = df['date'].dt.isocalendar().week

In [6]:
df.head(50)

,id,date,latitude,longitude,week
0,1,2015-03-04 00:35:00,0.527129,0.358134,10
1,1,2015-03-04 00:35:00,0.341051,0.329298,10
2,1,2015-03-04 00:35:00,0.705571,0.122502,10
3,1,2015-03-04 00:35:00,0.618180,0.567444,10
4,1,2015-03-04 00:35:00,0.686230,0.560744,10
5,1,2015-03-04 00:35:00,0.171775,0.197126,10
6,1,2015-03-04 00:35:00,0.572717,0.629575,10
7,1,2015-03-04 00:35:00,0.892403,0.299951,10
8,1,2015-03-04 00:35:00,0.880168,0.365618,10
9,1,2015-03-04 00:36:00,0.712761,0.360436,10


In [11]:
df.sample(frac=1).reset_index(drop=True)

,id,date,latitude,longitude,week
0,1,2015-03-04 00:41:00,0.759197,0.686576,10
1,1,2015-03-04 00:37:00,0.746124,0.654643,10
2,1,2015-03-04 00:42:00,0.430944,0.120310,10
3,1,2015-03-04 01:02:00,0.233012,0.387355,10
4,1,2015-03-04 00:37:00,0.050518,0.613878,10
...,...,...,...,...,...
494,1,2015-03-04 00:44:00,0.615615,0.319315,10
495,1,2015-03-04 01:01:00,0.681202,0.686642,10
496,1,2015-03-04 01:02:00,0.546555,0.262620,10
497,1,2015-03-04 00:38:00,0.240207,0.440551,10


In [14]:
# Save file
df.to_csv('../Data_Anonymisation_et_Reidentification/toscv.scv', index = False, header = False, columns = None, sep='\t')

In [107]:
# Change id : id = abs(10000 * tan(id (+ cos(week / 2) if week % 2 == 0) (- cos(week / 2) if week % 2 != 0))
df['new_column'] = df.apply(lambda row: int(10000 * (np.tan(row['id'] + np.cos(row['week'] / 2)) if row['week'] % 2 == 0 else np.tan(row['id'] - np.cos(row['week'] / 2)))), axis=1)
df

,id,date,latitude,longitude,week,new_column
0,1,2015-03-04 00:35:00,0.527129,0.358134,10,33864
1,1,2015-03-04 00:35:00,0.341051,0.329298,10,33864
2,1,2015-03-04 00:35:00,0.705571,0.122502,10,33864
3,1,2015-03-04 00:35:00,0.618180,0.567444,10,33864
4,1,2015-03-04 00:35:00,0.686230,0.560744,10,33864
...,...,...,...,...,...,...
494,1,2015-03-04 01:03:00,0.788060,0.225917,10,33864
495,1,2015-03-04 01:03:00,0.668119,0.718251,10,33864
496,1,2015-03-04 01:03:00,0.173801,0.770294,10,33864
497,1,2015-03-04 01:03:00,0.835937,0.000620,10,33864


In [108]:
#Random minute : returns a new date by adding or subtracting the random number of minutes, depending on whether the hour remains the same.
def random_minute(date):
    rand_min = random.randint(-29,29)
    return date - dt.timedelta(minutes=rand_min) if date.hour == (date - dt.timedelta(minutes=rand_min)).hour else date + dt.timedelta(minutes=rand_min)

#Change date to random (same week)
def change_date(date):
    rand_day = int(np.random.choice([-1,0,1], p=[0.02, 0.96, 0.02]))
    return date - dt.timedelta(days=rand_day) if date.isocalendar()[1] == (date - dt.timedelta(days=rand_day)).isocalendar()[1] else date + dt.timedelta(days=rand_day)

#Change latitude : latitude = latitude + 0.005 * cos(latitude)
def change_lat(latitude):
    df['latitude']  = df.apply(lambda row: row['latitude'] + np.cos(row['latitude']), axis=1)
    return df['latitude']

#Change longitude : longitude = longitude + 0.005 * sin(longitude)
def change_long(longitude):
    df['longitude']  = df.apply(lambda row: row['longitude'] + np.sin(row['longitude']), axis=1)
    return df['longitude']

In [109]:
df['latitude'] = df['latitude'] + 0.005 * np.cos(df['latitude'])
df['longitude'] = df['longitude'] + 0.005 * np.sin(df['longitude'])

In [110]:
df['date'] = df['date'].apply(random_minute)

In [111]:
df

,id,date,latitude,longitude,week,new_column
0,1,2015-03-04 00:55:00,0.531450,0.359886,10,33864
1,1,2015-03-04 00:53:00,0.345763,0.330915,10,33864
2,1,2015-03-04 00:32:00,0.709377,0.123113,10,33864
3,1,2015-03-04 00:09:00,0.622255,0.570132,10,33864
4,1,2015-03-04 00:11:00,0.690098,0.563403,10,33864
...,...,...,...,...,...,...
494,1,2015-03-04 01:13:00,0.791586,0.227037,10,33864
495,1,2015-03-04 01:00:00,0.672044,0.721541,10,33864
496,1,2015-03-04 01:09:00,0.178725,0.773776,10,33864
497,1,2015-03-04 01:26:00,0.839290,0.000623,10,33864


In [113]:
# Round lat, long to 4 decimals
df['latitude'] = df['latitude'].round(decimals=4)
df['longitude'] = df['longitude'].round(decimals=4)

In [114]:
#Set id to 'DEL' with lines duplicated
id_time = df.drop_duplicates(['id', 'date', 'latitude', 'longitude'])
print(id_time.shape)

index_not_change = np.array(id_time.index)
df.loc[~df.index.isin(index_not_change) , 'id'] = "DEL"

print(df.head())

(499, 6)
  id                date  latitude  longitude  week  new_column
0  1 2015-03-04 00:55:00    0.5315     0.3599    10       33864
1  1 2015-03-04 00:53:00    0.3458     0.3309    10       33864
2  1 2015-03-04 00:32:00    0.7094     0.1231    10       33864
3  1 2015-03-04 00:09:00    0.6223     0.5701    10       33864
4  1 2015-03-04 00:11:00    0.6901     0.5634    10       33864
